# 🧵 YOLOv11 ile Kumaş Kusur Tespiti (Optimal Ayarlar - Textile Defect Detection)

**Modifikasyon:** Optimal Veri Artırımı ve Hyperparameter Düzenlemeleri (Epoch: 20, Mosaic/Mixup Augmentation, Balanced Data).

## 1. Asama: Kurulum

In [ ]:
import os
import torch

!pip install ultralytics opencv-python-headless matplotlib tqdm pyyaml -q

import ultralytics
ultralytics.checks()

print(f'\n🖥️ GPU: {torch.cuda.get_device_name(0)}')
print(f'🔥 PyTorch: {torch.__version__}')
print(f'✅ {version} Kurulum Tamamlandi!')

## 2. Asama: Balanced & Enlarged Veri On Isleme

In [ ]:
import os
import shutil
import random
import xml.etree.ElementTree as ET
import cv2
from tqdm.notebook import tqdm

# --- DIZIN AYARLARI ---
RAW_IMAGES_DIR = '/kaggle/input/datasets/bnhphan/zju-dataset/ZJU-Leaper/Images'
RAW_XML_DIR    = '/kaggle/input/datasets/bnhphan/zju-dataset/ZJU-Leaper/Annotations/xmls'
OUTPUT_DIR     = '/kaggle/working/yolo_dataset'

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
for split in ['train', 'val']:
    for dtype in ['images', 'labels']:
        os.makedirs(f"{OUTPUT_DIR}/{split}/{dtype}", exist_ok=True)

def convert_zju_xml(xml_file, img_path):
    tree = ET.parse(xml_file)
    root = tree.getroot()
    defective_tag = root.find('defective')
    if defective_tag is None or int(defective_tag.text) == 0:
        return []

    img = cv2.imread(img_path)
    if img is None: return []
    h_img, w_img = img.shape[:2]
    
    yolo_lines = []
    for box in root.findall('bbox'):
        try:
            xmin, ymin, xmax, ymax = (float(box.find(k).text) for k in ['xmin', 'ymin', 'xmax', 'ymax'])
            x_center = min(1.0, max(0.0, ((xmin + xmax) / 2) / w_img))
            y_center = min(1.0, max(0.0, ((ymin + ymax) / 2) / h_img))
            w = min(1.0, max(0.0, (xmax - xmin) / w_img))
            h = min(1.0, max(0.0, (ymax - ymin) / h_img))
            yolo_lines.append(f"0 {x_center} {y_center} {w} {h}")
        except: continue
    return yolo_lines

all_images = sorted([f for f in os.listdir(RAW_IMAGES_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
print(f"📊 Toplam gorsel sayisi: {len(all_images)}")

defective_imgs, normal_imgs = [], []
for img_name in tqdm(all_images, desc="Gorseller taraniyor"):
    xml_path = os.path.join(RAW_XML_DIR, os.path.splitext(img_name)[0] + '.xml')
    if os.path.exists(xml_path):
        try:
            if int(ET.parse(xml_path).getroot().find('defective').text) == 1:
                defective_imgs.append(img_name)
                continue
        except: pass
    normal_imgs.append(img_name)

random.seed(42)
random.shuffle(defective_imgs)
random.shuffle(normal_imgs)

take_def = min(5000, len(defective_imgs))
take_norm = min(5000, len(normal_imgs))

selected_defective = defective_imgs[:take_def]
selected_normal = normal_imgs[:take_norm]
all_images = selected_defective + selected_normal
random.shuffle(all_images)

print(f"\n⚖️ Dengeli orneklem (OPTIMIZE): {len(selected_defective)} kusurlu + {len(selected_normal)} normal = {len(all_images)} toplam")

split_idx = int(len(all_images) * 0.8)
train_imgs, val_imgs = all_images[:split_idx], all_images[split_idx:]

def process_batch(image_list, split_type):
    defect_cnt = 0
    for img_name in tqdm(image_list, desc=f"{split_type} hazirlaniyor"):
        shutil.copy(os.path.join(RAW_IMAGES_DIR, img_name), os.path.join(OUTPUT_DIR, split_type, 'images', img_name))
        xml_path = os.path.join(RAW_XML_DIR, os.path.splitext(img_name)[0] + '.xml')
        label_path = os.path.join(OUTPUT_DIR, split_type, 'labels', os.path.splitext(img_name)[0] + '.txt')
        
        if os.path.exists(xml_path):
            data = convert_zju_xml(xml_path, os.path.join(RAW_IMAGES_DIR, img_name))
            with open(label_path, 'w') as f:
                if data:
                    f.write('\n'.join(data))
                    defect_cnt += 1
        else:
            open(label_path, 'w').close()
    print(f"✅ {split_type}: {len(image_list)} gorsel, {defect_cnt} kusurlu")

process_batch(train_imgs, 'train')
process_batch(val_imgs, 'val')

## 3. Asama: YAML Olusturma

In [ ]:
import yaml
data_yaml = {
    'train': '/kaggle/working/yolo_dataset/train/images',
    'val':   '/kaggle/working/yolo_dataset/val/images',
    'nc': 1, 'names': ['Defective']
}
with open('/kaggle/working/data.yaml', 'w') as f:
    yaml.dump(data_yaml, f)
print("✅ data.yaml olusturuldu!")

## 4. Asama: Optimize Edilmis Egitim

In [ ]:
import os
from ultralytics import YOLO
os.chdir('/kaggle/working')
os.environ['WANDB_DISABLED'] = 'true'

model = YOLO('yolo11n.pt')

# OPTIMAL AYARLAR ILE EGITIM
# epochs=20: Data miktari artirildigi icin egitim uzatıyor
# batch=32, imgsz=640: Mimariye uygun hiz & performans dengesi sağlıyor
# mixup, mosaic, label_smoothing: Overfitting onleme ve data augmentation (veri artırımı)
results = model.train(
    data='/kaggle/working/data.yaml',
    epochs=20,
    imgsz=640,
    batch=32,
    device=0,
    workers=8,
    name='yolov11_textile_opt',
    project='runs/train',
    optimizer='auto',  # Otomatik AdamW ya da SGD secer
    label_smoothing=0.1,
    patience=10,
    augment=True,
    mosaic=1.0,
    mixup=0.15,
    close_mosaic=5     # Son 5 epoch sirasinda mosaic ougmentation kapatilir ki model net detaylara odaklansin.
)


## 5. Asama: Inference & Gorsellestirme

In [ ]:
import glob, os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from ultralytics import YOLO
os.chdir('/kaggle/working')
best_w = 'runs/train/yolov11_textile_opt/weights/best.pt'
if os.path.exists(best_w):
    model = YOLO(best_w)
    model.predict(source='/kaggle/working/yolo_dataset/val/images', save=True, conf=0.15, project='runs/detect', name='test_opt_res')
    
    det_dir = max(glob.glob('runs/detect/test_opt_res*'), key=os.path.getmtime)
    imgs = sorted(glob.glob(f'{det_dir}/*.jpg') + glob.glob(f'{det_dir}/*.png'))[:12]
    if imgs:
        fig, axes = plt.subplots(3, 4, figsize=(20, 15))
        for i, ax in enumerate(axes.flat):
            if i < len(imgs):
                ax.imshow(mpimg.imread(imgs[i])); ax.axis('off')
        plt.suptitle(f'Optimized {version} Sonuclari', fontsize=16)
        plt.show()


## 6. Asama: Export (ONNX)

In [ ]:
import os
from IPython.display import FileLink
from ultralytics import YOLO
os.chdir('/kaggle/working')
best_w = 'runs/train/yolov11_textile_opt/weights/best.pt'
if os.path.exists(best_w):
    display(FileLink(best_w))
    model = YOLO(best_w)
    onnx_path = model.export(format='onnx', imgsz=640)
    if os.path.exists(onnx_path): display(FileLink(onnx_path))
